# 1

In [4]:
# -----######-----######-----######-----######-----######
# KICK ENHANCER ENGINE (1 BEAT / BPM LOCKED)
# -----######-----######-----######-----######-----######

import numpy as np
import librosa
import soundfile as sf
from scipy.signal import butter, lfilter
from tqdm import tqdm

# ----- LOW PASS FILTER (SUB SHAPING)
def _lowpass(x, sr, cutoff=120):
    nyq = 0.5 * sr
    norm = cutoff / nyq
    b, a = butter(4, norm, btype='low')
    return lfilter(b, a, x)

# ----- SOFT CLIPPER
def _soft_clip(x, drive=1.5):
    return np.tanh(drive * x)

# ----- TRANSIENT SHAPER (SIMPLE)
def _transient_boost(x, amount=1.5):
    diff = np.diff(np.concatenate([[0], x]))
    return x + amount * diff

# ----- MAIN FUNCTION
def _kick_0804_i1_GET_punchy_kick(
    path_in,
    path_out,
    bpm,
    sub_freq=50,
    sub_gain=0.4,
    transient_amt=1.5,
    drive=1.3
):
    """
    INPUT:
        path_in        : path to mp3/wav kick
        path_out       : output path (wav recommended)
        bpm            : target BPM (1 beat enforced)
        sub_freq       : sine reinforcement freq (Hz)
        sub_gain       : sub amplitude
        transient_amt  : punch strength
        drive          : saturation amount

    OUTPUT:
        exported enhanced kick
    """

    # LOAD
    y, sr = librosa.load(path_in, sr=None, mono=True)

    # ---- FORCE LENGTH = 1 BEAT
    beat_duration = 60 / bpm
    target_len = int(sr * beat_duration)

    if len(y) > target_len:
        y = y[:target_len]
    else:
        y = np.pad(y, (0, target_len - len(y)))

    # ---- TRANSIENT BOOST
    y = _transient_boost(y, transient_amt)

    # ---- SUB LAYER (SINE)
    t = np.linspace(0, beat_duration, target_len)
    sub = np.sin(2 * np.pi * sub_freq * t)

    # envelope (decay so it's not muddy)
    env = np.exp(-5 * t)
    sub = sub * env

    y = y + sub_gain * sub

    # ---- LOW END CLEANUP
    y = _lowpass(y, sr, cutoff=120)

    # ---- SATURATION
    y = _soft_clip(y, drive)

    # ---- NORMALIZE
    y = y / np.max(np.abs(y))

    # ---- EXPORT
    sf.write(path_out, y, sr)

    return path_out

In [6]:
# INPUTS
path_in  = "/Users/yerik/Music/_7_GENERT/_s13_d3-7140r-9AEmin---25-VARIOUS-0920srcdrumssil020-luS-2A-D#min-10B-Dmaj_11A-F#min-123.mp3"
path_out = "/Users/yerik/Music/_7_GENERT/_GNRT_4_.wav"

# RUN
_kick_0804_i1_GET_punchy_kick(
    path_in=path_in,
    path_out=path_out,
    bpm=128
)

'/Users/yerik/Music/_7_GENERT/_GNRT_2_.wav'

# 2

In [7]:
# -----######-----######-----######-----######-----######
# ULTIMATE KICK DESIGN ENGINE (FULL PARAM CONTROL)
# -----######-----######-----######-----######-----######

import numpy as np
import librosa
import soundfile as sf
from scipy.signal import butter, lfilter
from tqdm import tqdm

# ---------------- FILTERS ----------------
def _lowpass(x, sr, cutoff):
    b, a = butter(4, cutoff/(0.5*sr), btype='low')
    return lfilter(b, a, x)

def _highpass(x, sr, cutoff):
    b, a = butter(4, cutoff/(0.5*sr), btype='high')
    return lfilter(b, a, x)

def _bandpass(x, sr, low, high):
    b, a = butter(4, [low/(0.5*sr), high/(0.5*sr)], btype='band')
    return lfilter(b, a, x)

# ---------------- ENVELOPES ----------------
def _adsr(n, sr, attack, decay, sustain, release):
    a = int(sr * attack)
    d = int(sr * decay)
    r = int(sr * release)
    s = max(0, n - (a + d + r))

    env = np.concatenate([
        np.linspace(0, 1, a),
        np.linspace(1, sustain, d),
        np.ones(s) * sustain,
        np.linspace(sustain, 0, r)
    ])
    return env[:n]

# ---------------- SATURATION ----------------
def _sat(x, drive=1.0, mode="tanh"):
    if mode == "tanh":
        return np.tanh(drive * x)
    elif mode == "soft":
        return x / (1 + np.abs(drive * x))
    return x

# ---------------- COMP ----------------
def _comp(x, thresh=0.5, ratio=4):
    y = x.copy()
    mask = np.abs(y) > thresh
    y[mask] = np.sign(y[mask]) * (thresh + (np.abs(y[mask])-thresh)/ratio)
    return y

# ---------------- PITCH ENVELOPE ----------------
def _pitch_env(freq_start, freq_end, n):
    return np.linspace(freq_start, freq_end, n)

# ---------------- MAIN ----------------
def _kick_0804_i3_GET_ultimate_kick(
    path_in,
    path_out,
    bpm,

    # --- LENGTH / TIME ---
    beat_fraction=1.0,     # 1 beat, 0.5 = half beat
    tail_extend=0.0,       # extend tail (seconds)

    # --- SUB ---
    sub_freq=50,
    sub_gain=0.5,
    sub_decay=0.3,

    # --- BODY ---
    body_gain=1.0,
    body_low=100,
    body_high=2000,

    # --- CLICK ---
    click_gain=1.2,
    click_freq=3000,

    # --- ENVELOPE ---
    attack=0.001,
    decay=0.1,
    sustain=0.3,
    release=0.2,

    # --- PITCH ENV (HUGE FOR PUNCH) ---
    pitch_start=120,
    pitch_end=50,

    # --- SATURATION ---
    drive=1.4,
    sat_mode="tanh",

    # --- COMPRESSION ---
    comp_thresh=0.6,
    comp_ratio=4,

    # --- FINAL ---
    clip_drive=1.3,
    normalize=True
):
    """
    FULL CONTROL KICK DESIGN ENGINE
    """

    # LOAD
    y, sr = librosa.load(path_in, sr=None, mono=True)

    # ---- LENGTH CONTROL
    beat_len = int(sr * (60 / bpm) * beat_fraction)
    beat_len += int(sr * tail_extend)

    y = y[:beat_len] if len(y) > beat_len else np.pad(y, (0, beat_len-len(y)))

    n = len(y)
    t = np.arange(n) / sr

    # ---- MULTIBAND SPLIT
    sub_band  = _lowpass(y, sr, 120)
    body_band = _bandpass(y, sr, body_low, body_high)
    click_band = _highpass(y, sr, click_freq)

    # ---- PITCHED SUB (REAL KICK FEEL)
    freq_curve = _pitch_env(pitch_start, pitch_end, n)
    phase = 2 * np.pi * np.cumsum(freq_curve) / sr
    sub = np.sin(phase)

    # ---- SUB ENVELOPE
    sub_env = np.exp(-t / sub_decay)
    sub_layer = sub * sub_env * sub_gain

    # ---- CLICK ENHANCE
    click_band *= click_gain

    # ---- BODY GAIN
    body_band *= body_gain

    # ---- ADSR ON ORIGINAL
    env = _adsr(n, sr, attack, decay, sustain, release)
    y = y * env

    # ---- REBUILD
    y = sub_band + body_band + click_band + sub_layer

    # ---- SATURATION
    y = _sat(y, drive, sat_mode)

    # ---- COMPRESSION
    y = _comp(y, comp_thresh, comp_ratio)

    # ---- FINAL CLIP
    y = np.tanh(clip_drive * y)

    # ---- NORMALIZE
    if normalize:
        y = y / np.max(np.abs(y))

    # EXPORT
    sf.write(path_out, y, sr)

    return path_out

In [21]:

path_in  = '/Users/yerik/Music/_7_GENERT/_s13_d7-7150o-6AGmin---25-VARIOUS-0545srcdrumssil010-luO-9B-Gmaj-6A-Gmin_1A-G#min-161.mp3'
path_out = "/Users/yerik/Music/_7_GENERT/_GNRT_5_.wav"

_kick_0804_i3_GET_ultimate_kick(
    path_in=path_in,
    path_out=path_out,
    bpm=160,

    sub_freq=48,
    sub_gain=.5,

    pitch_start=170,
    pitch_end=45,

    attack=0.001,
    decay=0.08,
    sustain=0.2,
    release=0.25,

    drive=1.5,
    comp_ratio=5
)

'/Users/yerik/Music/_7_GENERT/_GNRT_5_.wav'

# 3

In [24]:


# -----######-----######-----######-----######-----######
# CLEAN KICK ENHANCER (HIGH-END / NO CLIPPING)
# -----######-----######-----######-----######-----######

import numpy as np
import librosa
import soundfile as sf
from scipy.signal import butter, lfilter
from tqdm import tqdm

# ---------------- FILTERS ----------------
def _lowpass(x, sr, cutoff):
    b, a = butter(2, cutoff/(0.5*sr), btype='low')
    return lfilter(b, a, x)

def _highpass(x, sr, cutoff):
    b, a = butter(2, cutoff/(0.5*sr), btype='high')
    return lfilter(b, a, x)

# ---------------- SAFE TRANSIENT ----------------
def _transient_refine(x, amt=0.2):
    dx = np.diff(np.concatenate([[0], x]))
    return x + amt * dx

# ---------------- SUB ALIGN ----------------
def _aligned_sub(y, sr, freq):
    peak = np.argmax(np.abs(y))
    t = np.arange(len(y)) / sr
    phase = -2*np.pi*freq*(peak/sr)
    return np.sin(2*np.pi*freq*t + phase)

# ---------------- ENVELOPE TIGHTEN ----------------
def _tighten_tail(x, strength=3.0):
    n = len(x)
    t = np.linspace(0, 1, n)
    env = np.exp(-strength * t)
    return x * env

# ---------------- SOFT HARMONIC (VERY LIGHT) ----------------
def _harmonic_enhance(x, amt=0.1):
    return x + amt * (x**3)

# ---------------- MAIN ----------------
def _kick_0804_i4_GET_clean_enhanced(
    path_in,
    path_out,
    bpm,

    # subtle controls
    sub_freq=50,
    sub_gain=0.15,

    transient_amt=0.15,
    tail_tightness=3.5,

    harmonic_amt=0.08,

    low_tighten=0.2,   # reduces mud
    normalize=True
):
    """
    CLEAN / HIGH-END kick enhancement
    no clipping, no distortion artifacts
    """

    # LOAD
    y, sr = librosa.load(path_in, sr=None, mono=True)

    # ---- FORCE 1 BEAT
    target_len = int(sr * (60 / bpm))
    y = y[:target_len] if len(y) > target_len else np.pad(y, (0, target_len-len(y)))

    # ---- SPLIT
    sub_band = _lowpass(y, sr, 120)
    top_band = _highpass(y, sr, 120)

    # ---- TRANSIENT (VERY CONTROLLED)
    y = _transient_refine(y, transient_amt)

    # ---- SUB (PHASE ALIGNED / LOW GAIN)
    sub = _aligned_sub(y, sr, sub_freq)
    sub_env = _tighten_tail(sub, tail_tightness)
    y = y + sub_gain * sub_env

    # ---- LOW END CONTROL (TIGHTEN)
    sub_band = sub_band * (1 - low_tighten)

    # ---- HARMONIC (VERY LIGHT)
    y = _harmonic_enhance(y, harmonic_amt)

    # ---- REBUILD
    y = sub_band + top_band

    # ---- FINAL NORMALIZE (NO CLIP)
    if normalize:
        peak = np.max(np.abs(y))
        if peak > 0:
            y = y / peak * 0.98  # headroom preserved

    # EXPORT
    sf.write(path_out, y, sr)

    return path_out


In [30]:
path_in  = '/Users/yerik/Music/_7_GENERT/_s13_d7-7150o-6AGmin---25-VARIOUS-0545srcdrumssil010-luO-9B-Gmaj-6A-Gmin_1A-G#min-161.mp3'
path_out = "/Users/yerik/Music/_7_GENERT/_GNRT_3_6_.wav"

_kick_0804_i4_GET_clean_enhanced(
    path_in=path_in,
    path_out=path_out,
    bpm=160,

    sub_freq=48,
    sub_gain=0.92,

    transient_amt=0.72,
    tail_tightness=9.8,

    harmonic_amt=8.06
)

'/Users/yerik/Music/_7_GENERT/_GNRT_3_6_.wav'